<a href="https://colab.research.google.com/github/luyov-belli/cv_experience_ocr_bert/blob/main/notebooks/colab_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# cv_experience_ocr_bert: etapas pesadas en Colab

Este notebook **no contiene lógica propia**: clona el repo y ejecuta los scripts de `src/`. Todo cambio de código se hace en la laptop, va a una rama y entra a `main` por PR. Aquí solo se hace `git pull`.

Las salidas pesadas (imágenes, OCR, pesos) se guardan en Google Drive mediante enlaces simbólicos, porque la máquina de Colab se borra al cerrar la sesión.

**Antes de empezar:**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4* (la descarga no la necesita, pero el OCR y BERT sí).
2. Panel izquierdo → 🔑 *Secrets* → crear `HF_TOKEN` con tu token de Hugging Face y activar *Notebook access*.

## 0. Preparación (correr siempre al abrir el notebook)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'Sin GPU'

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Token de Hugging Face: los Secrets solo se leen desde el kernel del notebook,
# no desde los scripts lanzados con `!python ...` (otro proceso). Se copia a una
# variable de entorno, que esos procesos heredan.
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('HF_TOKEN cargado:', bool(os.environ['HF_TOKEN']))

In [ ]:
import os

REPO_URL = 'https://github.com/luyov-belli/cv_experience_ocr_bert'
REPO = '/content/cv_experience_ocr_bert'
DRIVE = '/content/drive/MyDrive/cv_experience_ocr_bert'
BRANCH = 'main'  # para probar una rama antes de fusionarla, cambia esto

if not os.path.exists(REPO):
    !git clone -q {REPO_URL} {REPO}
%cd {REPO}
!git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
!git log --oneline -1

# Carpetas pesadas -> Drive (persisten entre sesiones)
for sub in ['data/raw', 'data/ocr', 'models/bert']:
    target = f'{DRIVE}/{sub}'
    os.makedirs(target, exist_ok=True)
    if not os.path.islink(sub):
        !rm -rf {sub}
        os.symlink(target, sub)
!ls -la data models

In [ ]:
# Dependencias (ver requirements-colab.txt). Se congelan en la versión que trae
# Colab solo las librerías de las que dependen sus herramientas internas:
#   pandas -> google-colab exige su versión exacta
#   numpy  -> numba y buena parte del stack de Colab
#   torch  -> build con CUDA de Colab
# pyarrow NO se congela: datasets 5.1 exige pyarrow>=24 y lo único en Colab que
# pide <24 es cudf, que este proyecto no usa.
# Si algo exigiera otra versión de las congeladas, pip falla aquí de forma
# explícita en vez de romper Colab en silencio.
import numpy, pandas, torch
keep = [f'{m.__name__}=={m.__version__.split("+")[0]}' for m in (numpy, pandas, torch)]
with open('/tmp/colab_keep.txt', 'w') as f:
    f.write('\n'.join(keep))
print('Se conservan:', keep)
!pip install -q -r requirements-colab.txt -c /tmp/colab_keep.txt

## 1. Descarga de datos (una sola vez)

Muestrea 800 CVs sin foto por ocupación (semilla 42) y guarda las imágenes en Drive. Toma del orden de 15–30 minutos. Si se corta, vuelve a correrla: el muestreo es determinista y produce la misma muestra.

In [ ]:
!python -m src.download_data

In [ ]:
# sampled_ids.csv vive en el repo (se versiona). Se copia a Drive para que
# lo bajes a la laptop y lo subas en un commit.
!cp data/sampled_ids.csv {DRIVE}/data/sampled_ids.csv
!ls {DRIVE}/data/raw | head; ls {DRIVE}/data/raw/images | wc -l

## 2. OCR con PaddleOCR (GPU)

**Correr en una sesión distinta a la de BERT.** PaddlePaddle-GPU y PyTorch traen cada uno sus propias librerías de NVIDIA, y al instalar una puede cambiar la versión que usa la otra. Para el OCR, abre una sesión nueva (*Entorno de ejecución → Desconectar y borrar*), corre la sección 0 y luego esta.

In [ ]:
# PaddlePaddle-GPU del mismo número de versión que requirements.txt (3.3.1), en el
# build que corresponde al CUDA que trae Colab.
import torch
major, minor = map(int, torch.version.cuda.split('.')[:2])
tag = 'cu130' if major >= 13 else 'cu129' if (major, minor) >= (12, 8) else 'cu126' if major == 12 else 'cu118'
print('CUDA de Colab:', torch.version.cuda, '-> índice', tag)
!pip install -q paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/{tag}/
!python -c "import paddle; print('paddle', paddle.__version__, '| GPUs:', paddle.device.cuda.device_count())"

### 2a. Elegir el modelo (decisión C)

Corre los 3 modelos (PP-OCRv5 server, PP-OCRv5 mobile, PP-OCRv6 medium) sobre los mismos 50 CVs (10 por ocupación) y mide tiempo, CER, WER y confianza contra el texto de referencia. No toca la caché de `data/ocr/`.

In [ ]:
!python -m src.ocr_extraction --compare 10
!mkdir -p {DRIVE}/output/metrics && cp output/metrics/ocr_model_comparison*.csv {DRIVE}/output/metrics/

### 2b. OCR de los 4,000 CVs

Pon en `OCR_MODEL` el modelo elegido en 2a (`server`, `mobile` o `v6`). Guarda un JSON por CV en Drive; si la sesión se corta, vuelve a correr la celda y retoma donde quedó.

In [ ]:
OCR_MODEL = 'v6'  # elegido en 2a: menor CER/WER y más rápido que v5 server
!python -m src.ocr_extraction --model {OCR_MODEL}

In [ ]:
# Métricas del OCR a Drive, y los 4,000 JSON en un solo zip para bajarlos a la laptop.
!cp output/metrics/ocr_stats.csv output/metrics/ocr_failures.csv output/metrics/ocr_run_summary.json {DRIVE}/output/metrics/
!cd data && zip -qr {DRIVE}/ocr_json.zip ocr && ls -lh {DRIVE}/ocr_json.zip
!ls {DRIVE}/data/ocr | wc -l

## 3. Fine-tuning de ModernBERT (GPU)

**Sesión distinta a la del OCR:** *Entorno de ejecución → Desconectar y borrar*, GPU T4, corre la sección 0 y luego esta (no hace falta instalar PaddlePaddle).

### 3a. Regenerar el texto limpio

El texto limpio solo vive en la laptop, pero Colab tiene en Drive todo lo necesario para regenerarlo (JSON del OCR, texto de referencia y splits del repo). La limpieza es determinista: la huella debe coincidir con la de la laptop.

In [ ]:
!python -m src.preprocessing > /tmp/prep.log && tail -n 3 /tmp/prep.log
import hashlib, pandas as pd
df = pd.read_parquet('data/processed/cv_text.parquet').sort_values('id')
h = hashlib.sha256('\x1e'.join(df.id + '\x1f' + df.text_clean + '\x1f' + df.text_ref_clean).encode('utf-8')).hexdigest()[:16]
print('huella:', h, '| esperada: 1ac233e7083a5694 |', 'OK' if h == '1ac233e7083a5694' else 'DISTINTA: avisar')

### 3b. Entrenar con MSE (modelo principal)

Hasta 10 épocas con early stopping por MAE de validación (paciencia 2). Los checkpoints intermedios van al disco local de Colab (`/content`, rápido); solo el mejor modelo se guarda en Drive (`models/bert/modernbert_mse`).

In [ ]:
!python -m src.train_bert --loss mse --ckpt-dir /content

### 3c. Entrenar con Huber (experimento de Q3)

Mismos hiperparámetros y semilla; solo cambia la pérdida (Huber con delta = 1 año).

In [ ]:
!python -m src.train_bert --loss huber --ckpt-dir /content

In [ ]:
# Predicciones y métricas a Drive (los pesos ya están en Drive vía models/bert).
!mkdir -p {DRIVE}/output/predictions {DRIVE}/output/metrics
!cp output/predictions/modernbert_*.csv {DRIVE}/output/predictions/
!cp output/metrics/modernbert_*_run.json {DRIVE}/output/metrics/
!ls -lh {DRIVE}/output/predictions {DRIVE}/output/metrics | grep modernbert
!du -sh {DRIVE}/models/bert/*